# MT19937: Statistical PRNG and Cryptographic Vulnerabilities

## Objective

To understand why a statistical pseudorandom number generator
(PRNG) is unsuitable for cryptographic applications by:

1. Implementing an MT19937-based stream cipher.
2. Recovering a weak 16-bit seed using a known-plaintext attack.
3. Predicting password reset tokens generated using timestamps.

## 1. Introduction

MT19937 is a pseudorandom number generator based on the
Mersenne Twister algorithm.

It produces a sequence of pseudorandom numbers using an
internal state.

Although it has good statistical properties, it is not
cryptographically secure because its output is predictable
under certain conditions.

In this assignment, we demonstrate two vulnerabilities:
- Weak seed recovery.
- Timestamp-based token prediction.

# Part A: MT19937 Stream Cipher

## 2. Stream Cipher Implementation

A stream cipher encrypts plaintext by XORing it with
a generated keystream.

Encryption:

C = P XOR K

Decryption:

P = C XOR K

Where:
- P = Plaintext
- C = Ciphertext
- K = Keystream

## 3. Known Plaintext Attack

We generate plaintext containing a random printable prefix
of 5 to 20 characters followed by 14 consecutive A characters.

The plaintext is encrypted using a randomly selected
16-bit seed.

An attacker knows that the plaintext contains 14 consecutive
A characters but does not know their location.

The attacker tries all possible 16-bit seeds to recover
the original plaintext and seed.

Seed search space:

2^16 = 65536 possible seeds.

# Part B: Predictable Password Reset Tokens

## 4. Timestamp-Based Token Generation

In this experiment, MT19937 is seeded using the current
Unix timestamp in seconds.

The generator produces 16 consecutive 8-bit outputs,
which form a 16-byte reset token.

## 5. Token Prediction Attack

An attacker knows the approximate token generation time.

The attacker tests timestamps within the interval:

T - 300 to T + 300

Total possible timestamps:

601

Each timestamp is tested as a candidate seed.
The generated token is compared with the observed token.

# Part C: Security Analysis

## 6. Why MT19937 Is Not Cryptographically Secure

MT19937 is designed for statistical randomness rather
than cryptographic security.

Its outputs can become predictable when an attacker
obtains sufficient information.

Using a small seed space makes brute-force recovery easy.

Using timestamps as seeds also makes prediction easier
because the possible seed range is limited.

Cryptographic applications should use a
Cryptographically Secure Pseudorandom Number Generator (CSPRNG).

In [3]:
import random
import secrets
import string
import time

In [4]:
def stream_cipher(data, seed):

    # Initialize MT19937 with the given seed
    rng = random.Random(seed)

    result = bytearray()

    # Generate one 8-bit keystream value per byte
    for byte in data:

        key_byte = rng.getrandbits(8)

        # XOR plaintext/ciphertext with keystream
        result.append(byte ^ key_byte)

    return bytes(result)

In [5]:
# Original message
message = b"Hello MT19937 Stream Cipher"

# Fixed seed for testing
seed = 12345

# Encryption
ciphertext = stream_cipher(message, seed)

# Decryption using the same function
decrypted = stream_cipher(ciphertext, seed)

print("Original Message :", message.decode())

print("Seed             :", seed)

print("Ciphertext (Hex) :", ciphertext.hex())

print("Decrypted Message:", decrypted.decode())

print("Encryption Successful:", ciphertext != message)

print("Decryption Successful:", decrypted == message)

Original Message : Hello MT19937 Stream Cipher
Seed             : 12345
Ciphertext (Hex) : 22de6ebdbced018e6fd70876a74f7a2b6dba0f2fafe345ece54a28
Decrypted Message: Hello MT19937 Stream Cipher
Encryption Successful: True
Decryption Successful: True


In [6]:
# Generate random prefix length between 5 and 20
prefix_length = secrets.randbelow(16) + 5

# Generate random printable prefix
characters = string.ascii_letters + string.digits

prefix = ''.join(
    secrets.choice(characters)
    for _ in range(prefix_length)
)

# Append 14 consecutive A characters
plaintext = (prefix + "A" * 14).encode()

# Generate random 16-bit seed
original_seed = secrets.randbelow(65536)

# Encrypt plaintext
ciphertext = stream_cipher(plaintext, original_seed)

print("Original Seed      :", original_seed)

print("Prefix Length      :", prefix_length)

print("Original Plaintext :", plaintext.decode())

print("Plaintext Length   :", len(plaintext), "bytes")

print("Ciphertext (Hex)   :", ciphertext.hex())

Original Seed      : 26538
Prefix Length      : 9
Original Plaintext : rZZtmtAHwAAAAAAAAAAAAAA
Plaintext Length   : 23 bytes
Ciphertext (Hex)   : 5224b4235498b2471368c82761b2eee453a82f32c1a95c


In [7]:
# Known plaintext pattern
known_pattern = b"A" * 14

# Initialize recovery variables
recovered_seed = None
recovered_plaintext = None

candidates_tested = 0

start_time = time.time()

# Try every possible 16-bit seed
for candidate_seed in range(65536):

    candidates_tested += 1

    # Decrypt using candidate seed
    decrypted_text = stream_cipher(ciphertext, candidate_seed)

    # Check whether known pattern exists
    if known_pattern in decrypted_text:

        recovered_seed = candidate_seed
        recovered_plaintext = decrypted_text

        break

end_time = time.time()

print("Candidates Tested :", candidates_tested)

print("Time Taken        :", round(end_time - start_time, 4), "seconds")

print("Recovered Seed    :", recovered_seed)

if recovered_seed is not None:
    print("Recovered Plaintext:", recovered_plaintext.decode())

Candidates Tested : 26539
Time Taken        : 0.384 seconds
Recovered Seed    : 26538
Recovered Plaintext: rZZtmtAHwAAAAAAAAAAAAAA


In [8]:
print("========== VERIFICATION ==========")

print("Original Seed :", original_seed)

print("Recovered Seed:", recovered_seed)

print("Seed Correct:", recovered_seed == original_seed)

print("Known Pattern Found:",
      known_pattern in recovered_plaintext)

print("Plaintext Correct:",
      recovered_plaintext == plaintext)

assert recovered_seed == original_seed
assert recovered_plaintext == plaintext

print("\nPART A COMPLETED SUCCESSFULLY!")

========== VERIFICATION ==========
Original Seed : 26538
Recovered Seed: 26538
Seed Correct: True
Known Pattern Found: True
Plaintext Correct: True

PART A COMPLETED SUCCESSFULLY!


# Part B: Predictable Password Reset Tokens

## 7. Token Generation

In this experiment, the current Unix timestamp is used as
the seed for MT19937.

The generator produces 16 consecutive 8-bit outputs.
These outputs are combined to create a 16-byte token.

In [9]:
# Generate current Unix timestamp in integer seconds
timestamp = int(time.time())

# Initialize MT19937 using timestamp
rng = random.Random(timestamp)

# Generate 16 consecutive 8-bit outputs
token_bytes = bytes(
    rng.getrandbits(8) for _ in range(16)
)

# Convert token into hexadecimal
observed_token = token_bytes.hex()

print("========== TOKEN GENERATION ==========")

print("Unix Timestamp :", timestamp)

print("Generated Token:", observed_token)

print("Token Length   :", len(token_bytes), "bytes")

========== TOKEN GENERATION ==========
Unix Timestamp : 1793859259
Generated Token: ce8cf2601f4f909b4186d5aa9976aecb
Token Length   : 16 bytes


## 8. Timestamp Prediction Attack

The attacker knows the observed token and the approximate
time at which it was generated.

The attacker tests all timestamps between T-300 and T+300.

For every candidate timestamp:
1. Initialize MT19937.
2. Generate 16 bytes.
3. Convert them into hexadecimal.
4. Compare with the observed token.

In [10]:
# Define search interval
start_timestamp = timestamp - 300
end_timestamp = timestamp + 300

total_candidates = end_timestamp - start_timestamp + 1

print("========== SEARCH INTERVAL ==========")

print("Start Timestamp :", start_timestamp)

print("End Timestamp   :", end_timestamp)

print("Total Candidates:", total_candidates)

========== SEARCH INTERVAL ==========
Start Timestamp : 1793858959
End Timestamp   : 1793859559
Total Candidates: 601


In [11]:
# Initialize recovery variables
recovered_timestamp = None
recovered_token = None

candidates_tested = 0

attack_start = time.time()

# Test every possible timestamp
for candidate_time in range(start_timestamp, end_timestamp + 1):

    candidates_tested += 1

    # Initialize MT19937 with candidate timestamp
    rng = random.Random(candidate_time)

    # Generate candidate 16-byte token
    candidate_token = bytes(
        rng.getrandbits(8) for _ in range(16)
    )

    # Compare with observed token
    if candidate_token.hex() == observed_token:

        recovered_timestamp = candidate_time
        recovered_token = candidate_token

attack_end = time.time()

print("========== ATTACK RESULTS ==========")

print("Observed Token     :", observed_token)

print("Candidates Tested  :", candidates_tested)

print("Time Taken         :",
      round(attack_end - attack_start, 6), "seconds")

print("Recovered Timestamp:", recovered_timestamp)

print("Recovered Token    :",
      recovered_token.hex() if recovered_token else None)

========== ATTACK RESULTS ==========
Observed Token     : ce8cf2601f4f909b4186d5aa9976aecb
Candidates Tested  : 601
Time Taken         : 0.023248 seconds
Recovered Timestamp: 1793859259
Recovered Token    : ce8cf2601f4f909b4186d5aa9976aecb


In [12]:
print("========== TOKEN VERIFICATION ==========")

# Generate token again using recovered timestamp
verification_rng = random.Random(recovered_timestamp)

verification_token = bytes(
    verification_rng.getrandbits(8) for _ in range(16)
)

print("Original Timestamp  :", timestamp)

print("Recovered Timestamp :", recovered_timestamp)

print("Original Token      :", observed_token)

print("Regenerated Token   :", verification_token.hex())

print("Timestamp Correct   :", recovered_timestamp == timestamp)

print("Token Matches       :", verification_token.hex() == observed_token)

assert recovered_timestamp == timestamp
assert verification_token.hex() == observed_token

print("\nPART B COMPLETED SUCCESSFULLY!")

========== TOKEN VERIFICATION ==========
Original Timestamp  : 1793859259
Recovered Timestamp : 1793859259
Original Token      : ce8cf2601f4f909b4186d5aa9976aecb
Regenerated Token   : ce8cf2601f4f909b4186d5aa9976aecb
Timestamp Correct   : True
Token Matches       : True

PART B COMPLETED SUCCESSFULLY!


# 9. Pseudocode

## A. MT19937 Stream Cipher Seed Recovery

1. Generate plaintext with a random prefix.
2. Append 14 consecutive A characters.
3. Select a random 16-bit seed.
4. Encrypt plaintext using MT19937.
5. For each candidate seed from 0 to 65535:
   a. Initialize MT19937 with candidate seed.
   b. Generate the keystream.
   c. Decrypt the ciphertext.
   d. Check for 14 consecutive A characters.
   e. If found, record the candidate seed and plaintext.
6. Verify the recovered seed and complete plaintext.

## B. Timestamp-Based Token Prediction

1. Generate current Unix timestamp T.
2. Initialize MT19937 using T.
3. Generate 16 consecutive 8-bit outputs.
4. Combine outputs into a 16-byte token.
5. Define the search interval T-300 to T+300.
6. For each candidate timestamp:
   a. Initialize MT19937 using candidate timestamp.
   b. Generate 16 bytes.
   c. Compare generated token with observed token.
   d. If they match, record the timestamp.
7. Regenerate the token using recovered timestamp.
8. Verify that both tokens match.

# 10. Conclusion

In this assignment, we demonstrated why MT19937 is unsuitable
for cryptographic applications.

In Part A, we implemented an MT19937-based stream cipher
and successfully recovered its 16-bit seed using a
known-plaintext attack.

In Part B, we generated a simulated password reset token
using the current Unix timestamp as the seed and recovered
the timestamp through brute-force searching.

These experiments demonstrate that statistical randomness
does not guarantee cryptographic security.

For secure applications, cryptographically secure
pseudorandom number generators (CSPRNGs) should be used
instead of MT19937.